# Session 2 — CI/CD for ML Systems

**Goal:** Understand how the manual workflow from Session 1 becomes an automated pipeline.

In Session 1 you manually:
1. Ran <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">04_mlflow_experiment.ipynb</span> → compared models
2. Ran <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">05_register_and_serve.ipynb</span> → deployed the best one

In production, this **must** happen automatically when:
- New training data arrives
- Code is merged to the main branch
- A scheduled retrain job triggers

**The tools:** Declarative Automation Bundles (DABs) + GitHub Actions

In [0]:
%run ../utils/config

## Declarative Automation Bundles
Formerly known as Databricks Asset Bundles, facilitate the adoption of software engineering best practices, including source control, code review, testing, and continuous integration and delivery (CI/CD)

Declarative Automation Bundles include:
-  Infrastructure and workspace configurations
-  Source files
-  Definitionsfor Databricks resources (i.e. Lakeflow Jobs, Lakeflow Spark Declarative Pipelines, Dashboards, Model Serving endpoints, MLflow Experiments, and MLflow registered models)
-  Unit tests and integration tests

<img src="../utils/resources/bundles-cicd.png">

Declarative Automation Bundles are typically developed in a local development environment such as VSCode or Cursor.  Databricks also supports [DABs in the Workspace](https://www.databricks.com/blog/announcing-databricks-asset-bundles-now-workspace) which enables you to build, edit and deploy using DABs from within a Databricks workspace.  We will use DABs in the Workspace in this workshop so we won't need a local dev environment.

## Navigate the Declarative Automation Bundle

1. Navigate to <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">../bundle</span> in the **Workspace Navigator**
2. Open <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">databricks.yml</span> — the root definition file - in a new browser tab.
3. Open <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">bundle/resources/retrain_job.yml</span> — defines the Lakeflow Job that automates the training pipeline from Session 1
4.  Click the rocket icon (<img src=../utils/resources/DeploymentIcon.png width=40px>) in the left sidebar to switch to the **Pipeline** authoring context.

This switches to the bundle editor.
1.  Confirm the Target is set to <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">dev</span>

## Deploy and Run the Retrain Pipeline

The bundle is pre-configured to target your personal schema automatically — no manual configuration needed.

1. Click **Deploy**, then **Deploy** again to confirm
2. Once the deploy finishes successfully, click the **Run job** icon next to the <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">[dev {username}] workshop_retrain_churn</span> job

<img src=../utils/resources/RunJob.png>

The first four tasks take about **4 min**. The final task, deploying the endpoint, takes an additional **8-10 min**.

The retrain job is also available in **Jobs & Pipelines** as <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">[dev {your_username}] workshop_retrain_churn</span>.

1. Click on <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">workshop_retrain_churn</span> in the bundle editor to open the job in a new tab.
2. The job runs asynchronously so you can continue on in the exercises, but use this UI to verify the job finishes successfully.

## The Continuous Integration / Continuous Deployment Workflow

Databricks Declarative Automation Bundles (DABs) work hand-in-hand with deployment automation tools such as Azure DevOps, GitHub Actions, or Jenkins.  In this workshop, we will use GitHub Actions as our primary example.

The CI/CD pipeline has two triggers:

**On Pull Request** (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">.github/workflows/pr_validation.yml</span>):
```
push to PR branch → run pytest → databricks bundle validate
```
Prevents merging broken code from reaching the main branch.  Whenever a Pull Request is issued, this workflow
1. runs unit tests (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">pytest</span>)
1. validates the bundle (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">databricks bundle validate</span>)

**On Merge to Main** (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">.github/workflows/deploy_retrain.yml</span>):
```
merge to main → bundle deploy → bundle run retrain_job
```
Automatically deploys and retrains after every approved code change.  Whenever code is merged back into the main branch, this workflow
1. runs unit tests (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">pytest</span>)
1. validates the bundle (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">databricks bundle validate</span>)
1. deploys the bundle (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">databricks bundle deploy --target prod</span>)
1. runs the job defined by the bundle (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">databricks bundle run workshop_retrain_churn -target prod</span>)

<img src="../utils/resources/CICD process.png">

## Discussion

- Where should the `DATABRICKS_TOKEN` come from? (Service principal, not a human user)
- What happens if the test fails? (PR is blocked from merging)
- What's the difference between `bundle validate` and `bundle deploy`?



➡️ Next: [02_simulate_drift.ipynb]($./02_simulate_drift) — inject realistic production drift into the inference log